# 🐄 Boeuf Tracker — Google Colab

Lance l'UI complète du **Boeuf Tracker** (YOLOv11 + DINOv2 + Re-ID) sur le GPU gratuit de Colab.

**Pourquoi Colab ?**
- 🎮 **GPU T4 (16 GB)** ou **A100 (40 GB)** — bien plus de VRAM qu'une GTX 1660 Ti (6 GB)
- 🧠 Modèles "lourds" supportés : `yolo11l-seg` + `dinov2-base` (impraticables en local)
- 🌐 UI accessible depuis n'importe quel navigateur via un tunnel **Cloudflare** (gratuit, sans compte)

**Prérequis**
1. Pousse ton repo sur GitHub (privé ou public)
2. Édite la cellule suivante : mets l'URL HTTPS dans `GIT_URL`
3. `Runtime` → `Change runtime type` → **T4 GPU** (ou A100)
4. Exécute les cellules dans l'ordre

À la fin, tu auras une URL publique `https://xxx.trycloudflare.com` pour ouvrir l'UI.

In [1]:
# =================================================================
# ⚙️  CONFIGURATION — modifie selon tes besoins
# =================================================================

# URL HTTPS de ton repo Git (obligatoire)
GIT_URL    = "https://github.com/ismaelgansonre/boeuf-tracker.git"
GIT_BRANCH = "fix/b36b1e9"
GIT_TOKEN  = ""

YOLO_MODEL  = "yolo11l-seg.pt"
DINO_MODEL  = "facebook/dinov2-base"

# --- Paramètres de détection ---
THRESHOLD   = 0.65     # Seuil cosine Re-ID (0.4 permissif → 0.8 strict)
CONF        = 0.4      # Confiance min YOLO (0.3 sensible → 0.6 strict)
IMGSZ       = 640      # Résolution YOLO (320 rapide → 1280 précis)
EMBED_EVERY = 10       # Re-embed tous les N frames (perf vs précision)

# --- Réseau ---
PORT = 5000

# --- Injection du token si repo privé ---
_repo_display = GIT_URL
if GIT_TOKEN and "@" not in GIT_URL and "github.com" in GIT_URL:
    GIT_URL = GIT_URL.replace("https://", f"https://x-access-token:{GIT_TOKEN}@")
    _repo_display = GIT_URL.replace(f"x-access-token:{GIT_TOKEN}@", "")

print(f"📦 Repo    : {_repo_display.replace('https://', '')}")
print(f"🤖 YOLO    : {YOLO_MODEL}")
print(f"🧠 DINOv2  : {DINO_MODEL}")
print(f"🔌 Port    : {PORT}")
print(f"⚙️  Settings: thr={THRESHOLD}  conf={CONF}  imgsz={IMGSZ}  embed_every={EMBED_EVERY}")

📦 Repo    : github.com/ismaelgansonre/boeuf-tracker.git
🤖 YOLO    : yolo11l-seg.pt
🧠 DINOv2  : facebook/dinov2-base
🔌 Port    : 5000
⚙️  Settings: thr=0.65  conf=0.4  imgsz=640  embed_every=10


In [2]:
import subprocess, sys

print("⏳ Installation des paquets système (ffmpeg)...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

print("⏳ Installation des dépendances Python...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.0.0",
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "transformers>=4.35.0",
    "opencv-python>=4.8.0",
    "numpy>=1.24.0",
    "Pillow>=10.0.0",
    "flask>=3.0.0",
    "werkzeug",
], check=True)

print("✅ Dépendances installées")

⏳ Installation des paquets système (ffmpeg)...
⏳ Installation des dépendances Python...
✅ Dépendances installées


In [ ]:
import os, subprocess
from pathlib import Path

# Garde-fou: GIT_URL pas modifié
if "USER" in GIT_URL or not GIT_URL.strip():
    raise SystemExit(
        "❌ Configure GIT_URL dans la cellule précédente !\n"
        "   Exemple: GIT_URL = 'https://github.com/ismaelgansonre/boeuf-tracker.git'"
    )

# Nom du dossier = dernier segment de l'URL
repo_name = GIT_URL.rstrip("/").split("/")[-1].replace(".git", "")
if "@" in repo_name:           # cas SSH git@github.com:user/repo.git
    repo_name = repo_name.split(":")[-1].split("/")[-1]

repo_path = Path("/content") / repo_name

if not repo_path.exists():
    print(f"⏳ Clonage du repo (branche {GIT_BRANCH})...")
    subprocess.run([
        "git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, str(repo_path)
    ], check=True)
    print("✅ Clone OK")
else:
    print(f"✅ Repo déjà présent: {repo_path}")
    print("   (supprime-le pour forcer un re-clone)")

os.chdir(repo_path)
print(f"📂 CWD = {os.getcwd()}")
print(f"📋 Contenu : {sorted(os.listdir('.'))[:15]}")

In [ ]:
import torch, subprocess
from pathlib import Path

if not torch.cuda.is_available():
    raise SystemExit(
        "❌ GPU non disponible !\n"
        "   Va dans Runtime → Change runtime type → Hardware accelerator → T4 GPU\n"
        "   Puis ré-exécute cette cellule."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU : {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")

# Avertissement VRAM si T4 et gros modèles
if "T4" in gpu_name.upper():
    if "l-seg" in YOLO_MODEL or "large" in DINO_MODEL.lower():
        print("⚠️  T4 détecté. Si OOM, baisse vers yolo11s-seg + dinov2-small dans la cellule de config.")

# Téléchargement des poids YOLO si absents
yolo_path = Path(YOLO_MODEL)
if not yolo_path.exists():
    print(f"⏳ Téléchargement de {YOLO_MODEL} (~50-100 MB)...")
    tag = "v8.3.0"
    url = f"https://github.com/ultralytics/assets/releases/download/{tag}/{YOLO_MODEL}"
    subprocess.run(["wget", "-q", "--show-progress", url], check=True)
    print("✅ Poids YOLO téléchargés")

# Liste des modèles présents
print("\n📦 Modèles .pt dans le dossier :")
for p in sorted(Path(".").glob("*.pt")):
    print(f"   - {p.name:25s}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
import subprocess, time, os, urllib.request, urllib.error
from pathlib import Path

# Nettoyage des runs précédents
subprocess.run(["pkill", "-f", "python app.py"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "cloudflared"],    stderr=subprocess.DEVNULL)
time.sleep(2)

# Commande Flask
flask_cmd = [
    "python", "app.py",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--device", "cuda:0",
    "--yolo-model", YOLO_MODEL,
    "--dino-model", DINO_MODEL,
    "--threshold", str(THRESHOLD),
    "--conf",      str(CONF),
    "--imgsz",     str(IMGSZ),
    "--embed-every", str(EMBED_EVERY),
]

# Logs dans /tmp/flask.log
log_path = Path("/tmp/flask.log")
log_path.unlink(missing_ok=True)
log_f = open(log_path, "wb", 0)  # unbuffered

print("⏳ Démarrage du serveur Flask...")
print("   (chargement YOLO + DINOv2 + DB → ~30-60s)")
print()
print("📜 Log en direct: !tail -f /tmp/flask.log\n")

flask_proc = subprocess.Popen(
    flask_cmd,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

# Attente que le serveur réponde (health-check sur /api/stats)
ready = False
for i in range(120):          # 4 minutes max
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats", timeout=1).read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError):
        if flask_proc.poll() is not None:
            print("❌ Le serveur a crashé. Dernières lignes du log :")
            subprocess.run(["tail", "-40", "/tmp/flask.log"])
            raise SystemExit(1)
        if i % 5 == 0:
            print(f"   ... chargement ({i*2}s)")

if not ready:
    print("❌ Timeout (4 min). Le serveur n'a pas démarré.")
    subprocess.run(["tail", "-60", "/tmp/flask.log"])
    raise SystemExit(1)

print(f"✅ Serveur Flask prêt sur http://127.0.0.1:{PORT}")
print(f"   PID Flask = {flask_proc.pid}")

# Vérif rapide: GPU et stats
try:
    import json as _j
    stats = _j.loads(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/stats").read())
    print(f"   Device actif : {stats.get('device')}")
    print(f"   YOLO chargé  : {stats.get('current', {}).get('yolo_model')}")
except Exception as e:
    print(f"   (stats non lisibles: {e})")

In [ ]:
import subprocess, re, time, os
from pathlib import Path

# Installation de cloudflared si pas déjà fait
if not Path("/usr/local/bin/cloudflared").exists() and not Path("/usr/bin/cloudflared").exists():
    print("⏳ Installation de cloudflared (~30 MB)...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        "-O", "/tmp/cloudflared.deb",
    ], check=True)
    subprocess.run(["dpkg", "-i", "/tmp/cloudflared.deb"], check=True)
    print("✅ cloudflared installé")

# Démarrage du tunnel
tunnel_log = Path("/tmp/tunnel.log")
tunnel_log.unlink(missing_ok=True)
tunnel_f = open(tunnel_log, "wb", 0)

print("⏳ Création du tunnel Cloudflare...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_f,
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

# Lecture de l'URL depuis les logs
url = None
for i in range(90):       # 3 min max
    time.sleep(2)
    content = tunnel_log.read_text(errors="ignore")
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', content)
    if m:
        url = m.group(1)
        break
    if tunnel_proc.poll() is not None:
        print("❌ Tunnel crashé. Log :")
        print(content[-1500:])
        raise SystemExit(1)

if not url:
    print("❌ Pas d'URL publique après 3 min. Log :")
    print(tunnel_log.read_text(errors="ignore")[-2000:])
    raise SystemExit(1)

# Affichage final grand format
print()
print("=" * 70)
print(f"  🌐  UI ACCESSIBLE À L'ADRESSE :")
print(f"      {url}")
print("=" * 70)
print()
print(f"  📺  Flux MJPEG  : {url}/video_feed")
print(f"  📊  Stats JSON  : {url}/api/stats")
print(f"  📋  Vidéos      : {url}/api/videos")
print(f"  🔧  Diagnostic  : {url}/api/diag")
print()
print("  ⚠️  Cette URL reste active tant que cette session Colab tourne.")
print("  ⚠️  Ouvre-la dans n'importe quel navigateur (même téléphone).")
print()
print("=" * 70)

In [ ]:
# OPTIONNEL: télécharge une vidéo d'exemple pour tester rapidement.
# Sinon, utilise le bouton "Upload" dans l'UI (upload direct navigateur → Colab).

SAMPLE_URL = ""   # ← colle ici une URL directe vers un .mp4 de bovins si tu veux
if SAMPLE_URL:
    print(f"⏳ Téléchargement de la vidéo d'exemple...")
    subprocess.run(["wget", "-q", "--show-progress", SAMPLE_URL, "-O", "sample_cattle.mp4"], check=True)
    size_mb = Path("sample_cattle.mp4").stat().st_size / 1024 / 1024
    print(f"✅ Vidéo téléchargée: sample_cattle.mp4 ({size_mb:.1f} MB)")
    print(f"   → Sélectionne-la dans le menu déroulant 'Source' de l'UI.")
else:
    print("💡 Pas de SAMPLE_URL configuré.")
    print("   → Upload ta vidéo via le bouton 'Upload' dans l'UI.")
    print("   → Ou mets une URL directe ci-dessus et ré-exécute.")

In [ ]:
# Surveille les logs en temps réel (Ctrl+C pour arrêter le monitoring).
# Le serveur Flask + le tunnel continuent à tourner en arrière-plan.

import IPython
from IPython.display import display, HTML

display(HTML("""
<div style="background:#1f2937;color:#10b981;padding:10px;border-radius:8px;
            font-family:monospace;border:1px solid #374151">
  ⏳ <b>Tail des logs Flask</b> — Ctrl+C pour arrêter ce monitoring<br>
  (le serveur et le tunnel restent actifs)
</div>
"""))

try:
    subprocess.run(["tail", "-f", "/tmp/flask.log"])
except KeyboardInterrupt:
    print("\n(Monitoring arrêté — serveur toujours up)")

## 🎉 C'est en ligne !

Ouvre l'URL affichée plus haut dans ton navigateur. Tu devrais voir :
- Le **flux MJPEG** en temps réel avec les silhouettes des bovins
- Les **sliders** pour ajuster seuil Re-ID, confiance, imgsz à chaud
- Le **journal** des événements (`NEW`, `MATCH`, `LOOP`, etc.)
- La **liste des animaux** détectés avec leur couleur stable

### 📹 Pour tester
1. Upload une vidéo de bovins via le bouton **Upload** dans l'UI
2. Le système détecte, tracke et identifie chaque bovin (`Boeuf_001`, `Boeuf_002`…)
3. Pendant la vidéo, clique sur **Re-match** si tu changes le seuil
4. La base est sauvegardée dans `cattle_db.pkl` → les bovins sont **reconnus** d'une vidéo à l'autre

### 🛑 Pour arrêter
- Ferme simplement l'onglet Colab, ou
- `Runtime` → `Manage sessions` → Terminate
- Le serveur Flask + le tunnel seront tués proprement

### 🔧 Troubleshooting
| Problème | Solution |
|---|---|
| GPU non dispo | `Runtime` → `Change runtime type` → **T4 GPU** |
| OOM CUDA | Baisse `YOLO_MODEL` à `yolo11s-seg.pt` et `DINO_MODEL` à `dinov2-small` |
| Tunnel mort | Ré-exécute la cellule 7 (nouvelle URL) |
| Re-ID trop strict | Baisse `THRESHOLD` à 0.50 dans la cellule de config puis redémarre |
| Trop de faux positifs | Monte `CONF` à 0.5 |
| Vidéo pas reconnue | Upload via le bouton **Upload** dans l'UI |